# Metrics-only pipeline (no model loading)

Recomputes all epistasis metrics from **existing embedding DBs only**. No models are loaded.

**Steps:**
1. Auto-discover which `(source, model)` pairs have DBs under `OUTPUT_BASE`
2. Compute **global** inverse covariance from `COV_INV_SOURCE_NAMES` (cached to `.npz`)
3. Compute **distance-binned** inverse covariance (cached to `.npz`)
4. Recompute metrics with global cov_inv (cached per-model to `.parquet`)
5. Recompute metrics with binned cov_inv (cached per-model to `.parquet`)
6. Merge global + binned into combined parquets

All intermediate results are cached to disk. If the notebook dies, re-run and it picks up where it left off.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Config
# ---------------------------------------------------------------------------
import sys, os, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "notebooks" / "paper_data_config.py").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.processing.pipeline_config import (
    SOURCE_COL, ID_COL, COV_INV_SOURCE_NAMES,
    get_single_dataframe_path,
)
from notebooks.processing.process_epistasis import FULL_MODEL_CONFIG
from notebooks.paper_data_config import EPISTASIS_PAPER_ROOT, data_dir, embeddings_dir

OUTPUT_BASE = embeddings_dir()
SHEETS_DIR = OUTPUT_BASE / "sheets"
CACHE_DIR  = OUTPUT_BASE / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SHEETS_DIR.mkdir(parents=True, exist_ok=True)

# Force recompute even if cache exists? Set True to ignore caches.
FORCE = False

print(f"OUTPUT_BASE = {OUTPUT_BASE}")
print(f"SHEETS_DIR  = {SHEETS_DIR}")
print(f"CACHE_DIR   = {CACHE_DIR}")
print(f"COV_INV_SOURCE_NAMES = {COV_INV_SOURCE_NAMES}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: Load dataframe & auto-discover available (source, model) DBs
# ---------------------------------------------------------------------------
import pandas as pd

single_path = get_single_dataframe_path(data_dir)
if single_path is None:
    raise FileNotFoundError("No single dataframe found.")
df_all = pd.read_csv(single_path, sep=None, engine="python")
print(f"Loaded {len(df_all)} rows, sources: {df_all[SOURCE_COL].unique().tolist()}")

# Scan OUTPUT_BASE for all existing .db files
available = {}  # model_key -> set of source_names
all_sources = sorted(d.name for d in OUTPUT_BASE.iterdir() if d.is_dir() and not d.name.startswith(("sheets", "cache", "null_cov", ".")))
for source_name in all_sources:
    source_dir = OUTPUT_BASE / source_name
    for db_file in source_dir.glob("*.db"):
        model_key = db_file.stem
        if model_key in FULL_MODEL_CONFIG:
            available.setdefault(model_key, set()).add(source_name)

MODEL_KEYS = sorted(available.keys())
SOURCE_NAMES = sorted(set().union(*available.values()))

print(f"\nDiscovered {len(MODEL_KEYS)} models across {len(SOURCE_NAMES)} sources:")
for mk in MODEL_KEYS:
    print(f"  {mk}: {sorted(available[mk])}")

# Filter df to only discovered sources
df_all = df_all[df_all[SOURCE_COL].isin(SOURCE_NAMES)].reset_index(drop=True)
print(f"\nFiltered dataframe: {len(df_all)} rows")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Compute global cov_inv (parallel across models, cached to npz)
# ---------------------------------------------------------------------------
import numpy as np
import multiprocessing as mp
from genebeddings.epistasis_features import compute_cov_inv_from_paths_combined
from notebooks.processing.process_epistasis import FULL_MODEL_CONFIG

cov_inv_sources = [s for s in COV_INV_SOURCE_NAMES if s in SOURCE_NAMES]
if not cov_inv_sources:
    raise ValueError(f"None of COV_INV_SOURCE_NAMES={COV_INV_SOURCE_NAMES} found in available sources={SOURCE_NAMES}")

df_cov = df_all[df_all[SOURCE_COL].isin(cov_inv_sources)].reset_index(drop=True)
cov_epi_ids = df_cov[ID_COL].dropna().astype(str).unique().tolist()
models_for_cov = [mk for mk in MODEL_KEYS if any(s in available[mk] for s in cov_inv_sources)]

# Check cache
cov_inv_by_model = {}
models_to_compute = []
for mk in models_for_cov:
    cache_path = CACHE_DIR / f"{mk}_cov_inv.npz"
    if cache_path.exists() and not FORCE:
        data = np.load(cache_path)
        cov_inv_by_model[mk] = (data["cov"], data["cov_inv"])
        print(f"  [cache hit] {mk}")
    else:
        models_to_compute.append(mk)

_out_base_cov = OUTPUT_BASE
_cov_sources = cov_inv_sources
_cov_epi_ids = cov_epi_ids
_cache_dir = CACHE_DIR

def _worker_cov_inv(model_key):
    """Compute global cov_inv for one model."""
    paths = []
    for src in _cov_sources:
        p = _out_base_cov / src / f"{model_key}.db"
        if p.exists():
            paths.append(str(p))
    if not paths:
        return model_key, None
    try:
        cov, cov_inv = compute_cov_inv_from_paths_combined(
            paths, epistasis_ids=_cov_epi_ids, method="ledoit_wolf", show_progress=False,
        )
        cache_path = _cache_dir / f"{model_key}_cov_inv.npz"
        np.savez(cache_path, cov=cov, cov_inv=cov_inv)
        return model_key, (cov, cov_inv)
    except Exception as e:
        return model_key, str(e)

if models_to_compute:
    n = min(len(models_to_compute), mp.cpu_count(), 16)
    print(f"Computing global cov_inv for {len(models_to_compute)} models with {n} workers...")
    with mp.Pool(n) as pool:
        for mk, result in pool.imap_unordered(_worker_cov_inv, models_to_compute):
            if result is None:
                print(f"  [no DBs] {mk}")
            elif isinstance(result, str):
                print(f"  [error] {mk}: {result}")
            else:
                cov_inv_by_model[mk] = result
                print(f"  [done] {mk}")

print(f"\nGlobal cov_inv for {len(cov_inv_by_model)} models: {sorted(cov_inv_by_model.keys())}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: Compute distance-binned cov_inv (parallel across models, cached to npz)
# ---------------------------------------------------------------------------
from genebeddings.genebeddings import parse_epistasis_id as _parse_epi

BIN_EDGES = (0, 1_000, 10_000, 100_000, float("inf"))
BIN_LABELS = ("0-1kb", "1-10kb", "10-100kb", "100kb+")

# Pre-compute per-bin epistasis ID lists (shared across workers)
_all_cov_ids = cov_epi_ids
_id_to_dist = {}
for eid in _all_cov_ids:
    try:
        (_, p1, _, _, _), (_, p2, _, _, _) = _parse_epi(eid)
        _id_to_dist[eid] = abs(p2 - p1)
    except Exception:
        pass

_dist_series = pd.Series(_id_to_dist, name="distance")
_bin_ser = pd.cut(_dist_series, bins=list(BIN_EDGES), labels=list(BIN_LABELS), right=False)
_bin_to_ids = {}
for label in BIN_LABELS:
    ids_in_bin = _bin_ser[_bin_ser == label].index.tolist()
    if len(ids_in_bin) >= 50:
        _bin_to_ids[label] = ids_in_bin

# Check cache
cov_inv_by_dist = {}
bins_to_compute = set()
for mk in models_for_cov:
    all_cached = True
    for label in list(BIN_LABELS) + ["global"]:
        safe_label = label.replace("+", "plus")
        cache_path = CACHE_DIR / f"{mk}_cov_inv_{safe_label}.npz"
        if cache_path.exists() and not FORCE:
            data = np.load(cache_path)
            cov_inv_by_dist.setdefault(label, {})[mk] = (data["cov"], data["cov_inv"])
        else:
            all_cached = False
    if all_cached:
        print(f"  [cache hit] {mk} (all bins)")
    else:
        bins_to_compute.add(mk)

def _worker_binned_cov(model_key):
    """Compute global + per-bin cov_inv for one model."""
    paths = []
    for src in _cov_sources:
        p = _out_base_cov / src / f"{model_key}.db"
        if p.exists():
            paths.append(str(p))
    if not paths:
        return model_key, None
    results = {}
    try:
        # Global
        cov, cov_inv = compute_cov_inv_from_paths_combined(
            paths, epistasis_ids=_all_cov_ids, method="ledoit_wolf", show_progress=False,
        )
        results["global"] = (cov, cov_inv)
        safe = "global"
        np.savez(_cache_dir / f"{model_key}_cov_inv_{safe}.npz", cov=cov, cov_inv=cov_inv)
        # Per-bin
        for label, bin_ids in _bin_to_ids.items():
            cov_b, cov_inv_b = compute_cov_inv_from_paths_combined(
                paths, epistasis_ids=bin_ids, method="ledoit_wolf", show_progress=False,
            )
            results[label] = (cov_b, cov_inv_b)
            safe = label.replace("+", "plus")
            np.savez(_cache_dir / f"{model_key}_cov_inv_{safe}.npz", cov=cov_b, cov_inv=cov_inv_b)
        return model_key, results
    except Exception as e:
        return model_key, str(e)

if bins_to_compute:
    models_fresh = sorted(bins_to_compute)
    n = min(len(models_fresh), mp.cpu_count(), 16)
    print(f"Computing binned cov_inv for {len(models_fresh)} models with {n} workers...")
    with mp.Pool(n) as pool:
        for mk, result in pool.imap_unordered(_worker_binned_cov, models_fresh):
            if result is None:
                print(f"  [no DBs] {mk}")
            elif isinstance(result, str):
                print(f"  [error] {mk}: {result}")
            else:
                for label, tup in result.items():
                    cov_inv_by_dist.setdefault(label, {})[mk] = tup
                print(f"  [done] {mk} ({len(result)} bins)")

print(f"\nBinned cov_inv bins: {[k for k in cov_inv_by_dist if k != 'global']}")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: Recompute metrics with global cov_inv (parallel across models)
# ---------------------------------------------------------------------------
import multiprocessing as mp
from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import add_epistasis_metrics

N_WORKERS = min(len(cov_inv_by_model), mp.cpu_count(), 16)

# Pre-split df by source (shared via fork, copy-on-write)
_source_dfs = {
    sn: df_all.loc[df_all[SOURCE_COL].astype(str) == sn].copy()
    for sn in df_all[SOURCE_COL].dropna().astype(str).unique()
}
_cov_inv_map = cov_inv_by_model
_out_base = OUTPUT_BASE
_id_col = ID_COL

def _worker_global(model_key):
    """Process all sources for one model. Runs in forked child."""
    cov, cov_inv = _cov_inv_map[model_key]
    context = FULL_MODEL_CONFIG[model_key][0]
    parts = []
    for source_name, df_s in _source_dfs.items():
        db_path = _out_base / source_name / f"{model_key}.db"
        if not db_path.exists() or len(df_s) == 0:
            continue
        db = VariantEmbeddingDB(str(db_path))
        try:
            # Dim check
            ci = cov_inv
            row = db.conn.execute("SELECT mut_id FROM embeddings LIMIT 1").fetchone()
            if row:
                emb_dim = len(db.load(row[0], as_torch=False))
                if emb_dim != cov_inv.shape[0]:
                    ci = None
            annotated = add_epistasis_metrics(
                df_s, db, model=None, id_col=_id_col,
                context=context, cov_inv=ci,
                force=False, batch_size=1, show_progress=False,
            )
            parts.append(annotated)
        finally:
            db.close()
    if parts:
        return model_key, pd.concat(parts, axis=0).sort_index()
    return model_key, None

# Figure out which models need processing
metrics_by_model = {}
models_cached = []
models_to_run = []
for mk in sorted(cov_inv_by_model.keys()):
    cache_path = SHEETS_DIR / f"epistasis_metrics_{mk}.parquet"
    if cache_path.exists() and not FORCE:
        metrics_by_model[mk] = pd.read_parquet(cache_path)
        models_cached.append(mk)
    else:
        models_to_run.append(mk)

if models_cached:
    print(f"[cache hit] {len(models_cached)} models: {models_cached}")

if models_to_run:
    n = min(N_WORKERS, len(models_to_run))
    print(f"Processing {len(models_to_run)} models with {n} workers...")
    with mp.Pool(n) as pool:
        for model_key, tbl in pool.imap_unordered(_worker_global, models_to_run):
            if tbl is not None:
                cache_path = SHEETS_DIR / f"epistasis_metrics_{model_key}.parquet"
                tbl.to_parquet(cache_path, index=False)
                metrics_by_model[model_key] = tbl
                print(f"  [done] {model_key}: {len(tbl)} rows")
            else:
                print(f"  [empty] {model_key}")

print(f"\nGlobal metrics for {len(metrics_by_model)} models")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: Recompute metrics with distance-binned cov_inv (parallel across models)
# ---------------------------------------------------------------------------
from genebeddings.genebeddings import parse_epistasis_id

# Assign distance bins to all rows
def _get_distance(eid):
    try:
        (_, pos1, _, _, _), (_, pos2, _, _, _) = parse_epistasis_id(str(eid))
        return abs(pos2 - pos1)
    except Exception:
        return float("nan")

distances = df_all[ID_COL].map(_get_distance)
df_all_binned = df_all.copy()
df_all_binned["distance_bin"] = pd.cut(distances, bins=list(BIN_EDGES), labels=list(BIN_LABELS), right=False)

global_cov_inv = cov_inv_by_dist.get("global", {})
model_keys_binned = sorted(global_cov_inv.keys())

# Pre-split binned df by source (shared via fork)
_source_dfs_binned = {
    sn: df_all_binned.loc[df_all_binned[SOURCE_COL].astype(str) == sn].copy()
    for sn in df_all_binned[SOURCE_COL].dropna().astype(str).unique()
}
_cov_inv_by_dist = cov_inv_by_dist
_global_cov_inv = global_cov_inv
_bin_labels = BIN_LABELS

def _worker_binned(model_key):
    """Process all sources and bins for one model. Runs in forked child."""
    context = FULL_MODEL_CONFIG[model_key][0]
    parts = []
    for source_name, df_s in _source_dfs_binned.items():
        db_path = _out_base / source_name / f"{model_key}.db"
        if not db_path.exists() or len(df_s) == 0:
            continue
        db = VariantEmbeddingDB(str(db_path))
        try:
            # Probe dim once per DB
            _db_dim = None
            row = db.conn.execute("SELECT mut_id FROM embeddings LIMIT 1").fetchone()
            if row:
                _db_dim = len(db.load(row[0], as_torch=False))

            bin_parts = []
            for label in _bin_labels:
                df_bin = df_s.loc[df_s["distance_bin"] == label]
                if len(df_bin) == 0:
                    continue
                if label in _cov_inv_by_dist and model_key in _cov_inv_by_dist[label]:
                    _, ci = _cov_inv_by_dist[label][model_key]
                elif model_key in _global_cov_inv:
                    _, ci = _global_cov_inv[model_key]
                else:
                    continue
                if _db_dim is not None and ci.shape[0] != _db_dim:
                    ci = None
                annotated_bin = add_epistasis_metrics(
                    df_bin, db, model=None, id_col=_id_col,
                    context=context, cov_inv=ci,
                    force=False, batch_size=1, show_progress=False,
                )
                bin_parts.append(annotated_bin)

            # NaN distance rows -> global
            df_nan = df_s.loc[df_s["distance_bin"].isna()]
            if len(df_nan) > 0 and model_key in _global_cov_inv:
                _, ci = _global_cov_inv[model_key]
                if _db_dim is not None and ci.shape[0] != _db_dim:
                    ci = None
                annotated_nan = add_epistasis_metrics(
                    df_nan, db, model=None, id_col=_id_col,
                    context=context, cov_inv=ci,
                    force=False, batch_size=1, show_progress=False,
                )
                bin_parts.append(annotated_nan)

            if bin_parts:
                parts.append(pd.concat(bin_parts, axis=0))
        finally:
            db.close()
    if parts:
        return model_key, pd.concat(parts, axis=0).sort_index()
    return model_key, None

# Figure out which models need processing
metrics_by_model_dist = {}
models_cached = []
models_to_run = []
for mk in model_keys_binned:
    cache_path = SHEETS_DIR / f"epistasis_metrics_{mk}_binned.parquet"
    if cache_path.exists() and not FORCE:
        metrics_by_model_dist[mk] = pd.read_parquet(cache_path)
        models_cached.append(mk)
    else:
        models_to_run.append(mk)

if models_cached:
    print(f"[cache hit] {len(models_cached)} models: {models_cached}")

if models_to_run:
    n = min(N_WORKERS, len(models_to_run))
    print(f"Processing {len(models_to_run)} models with {n} workers...")
    with mp.Pool(n) as pool:
        for model_key, tbl in pool.imap_unordered(_worker_binned, models_to_run):
            if tbl is not None:
                cache_path = SHEETS_DIR / f"epistasis_metrics_{model_key}_binned.parquet"
                tbl.to_parquet(cache_path, index=False)
                metrics_by_model_dist[model_key] = tbl
                print(f"  [done] {model_key}: {len(tbl)} rows")
            else:
                print(f"  [empty] {model_key}")

print(f"\nBinned metrics for {len(metrics_by_model_dist)} models")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7: Merge global + binned into combined parquets
# ---------------------------------------------------------------------------
MAHAL_COLS = ["epi_mahal", "mahal_obs", "mahal_add", "mahal_ratio", "log_mahal_ratio"]

all_model_keys = set(list(metrics_by_model) + list(metrics_by_model_dist))
merged_by_model = {}

for model_key in sorted(all_model_keys):
    df_global = metrics_by_model.get(model_key)
    df_binned = metrics_by_model_dist.get(model_key)

    if df_global is None and df_binned is None:
        continue

    # Start from global: rename mahal cols to global_*
    if df_global is not None:
        base = df_global.copy()
        mahal_present = [c for c in MAHAL_COLS if c in base.columns]
        base = base.rename(columns={c: f"global_{c}" for c in mahal_present})
    else:
        base = df_binned.drop(columns=MAHAL_COLS + ["distance_bin"], errors="ignore").copy()

    # Add binned mahal cols + distance_bin via merge (handles different row counts)
    if df_binned is not None:
        mahal_present_b = [c for c in MAHAL_COLS if c in df_binned.columns]
        binned_rename = {c: f"binned_{c}" for c in mahal_present_b}
        cols_to_merge = list(binned_rename.keys()) + ["distance_bin"]
        cols_to_merge = [c for c in cols_to_merge if c in df_binned.columns]
        df_b = df_binned[[ID_COL, SOURCE_COL] + cols_to_merge].rename(columns=binned_rename)
        base = base.merge(df_b, on=[ID_COL, SOURCE_COL], how="left")

    merged_by_model[model_key] = base

# Save combined
for model_key, tbl in merged_by_model.items():
    out = SHEETS_DIR / f"epistasis_metrics_{model_key}_combined.parquet"
    tbl.to_parquet(out, index=False)

    raw_cols = [c for c in tbl.columns if not c.startswith(("global_", "binned_")) and c != "distance_bin"]
    global_cols = [c for c in tbl.columns if c.startswith("global_")]
    binned_cols = [c for c in tbl.columns if c.startswith("binned_")]
    print(f"  {model_key}: {tbl.shape} -> {out.name}")
    print(f"    Raw: {len(raw_cols)}, Global mahal: {len(global_cols)}, Binned mahal: {len(binned_cols)}")

print(f"\nDone! {len(merged_by_model)} combined parquets saved to {SHEETS_DIR}")